# Week 4 Assignment - CIFAR-10 Image Classification (PyTorch)

## ANN vs CNN: Comparing Architectures & Training Strategies

**Author:** Sahil Yadav  
**Internship:** Celebal Technologies - Data Science  
**Dataset:** [CIFAR-10](https://www.cs.toronto.edu/~kriz/cifar.html) (auto-downloaded via `torchvision.datasets`)  

---

### Objective
Build image classification models on the **CIFAR-10 dataset** using both an **Artificial Neural Network (ANN)** and a **Convolutional Neural Network (CNN)**, then compare their performance across different architectures and training strategies.

### Approach
1. Load & explore the CIFAR-10 dataset
2. Preprocess - define transforms, DataLoaders
3. Build & train a baseline **ANN** model
4. Build & train a baseline **CNN** model
5. Compare accuracy, loss curves, and generalization
6. Upgrade CNN with **data augmentation**, **LR scheduling**, **EarlyStopping**
7. Final comparison table + conclusions


---
## 1. Install & Import Libraries


In [ ]:
!pip install -q torch torchvision matplotlib numpy pandas seaborn scikit-learn


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import DataLoader, random_split
import torchvision
import torchvision.transforms as transforms

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import warnings
warnings.filterwarnings('ignore')

sns.set(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

# Set device to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("PyTorch version:", torch.__version__)
print("Using device:", device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


---
## 2. Load & Explore CIFAR-10

CIFAR-10 has **60,000** colour images of size **32x32x3** split across 10 classes:  
Airplane, Automobile, Bird, Cat, Deer, Dog, Frog, Horse, Ship, Truck.

- 50,000 training images  
- 10,000 test images

The dataset is automatically downloaded by torchvision - no manual download needed.


In [ ]:
# Basic transform: ToTensor scales pixels to [0, 1]
basic_transform = transforms.Compose([
    transforms.ToTensor(),
])

# Download and load datasets
full_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=basic_transform)
test_dataset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=basic_transform)

class_names = full_train_dataset.classes

print(f"Total training set size: {len(full_train_dataset)}")
print(f"Test set size: {len(test_dataset)}")


### 2.1 Visualize Sample Images


In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(12, 5))
indices = np.random.choice(len(full_train_dataset), 10, replace=False)

for i, ax in enumerate(axes.flat):
    img, label = full_train_dataset[indices[i]]
    # PyTorch images are (C, H, W). We need (H, W, C) for matplotlib
    img = np.transpose(img.numpy(), (1, 2, 0))
    ax.imshow(img)
    ax.set_title(class_names[label], fontsize=11)
    ax.axis("off")

plt.suptitle("Random CIFAR-10 Training Samples", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()


### 2.2 Class Distribution
Each class should have roughly 5,000 samples (balanced dataset).


In [ ]:
targets = np.array(full_train_dataset.targets)
unique, counts = np.unique(targets, return_counts=True)

plt.figure(figsize=(10, 4))
sns.barplot(x=[class_names[i] for i in unique], y=counts, palette='viridis')
plt.title("Training Set - Class Distribution")
plt.ylabel("Count")
plt.xlabel("Class")
for i, v in enumerate(counts):
    plt.text(i, v + 50, str(v), ha='center', fontsize=9)
plt.tight_layout()
plt.show()
print("Classes are balanced!" if len(set(counts)) == 1 else f"Class counts: {dict(zip([class_names[i] for i in unique], counts))}")


---
## 3. Preprocessing (DataLoaders)

We'll split the 50,000 training images into 45,000 for training and 5,000 for validation.
We'll create DataLoaders to fetch data in batches of 64.


In [ ]:
# Split train into train and val
val_size = 5000
train_size = len(full_train_dataset) - val_size
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size])

batch_size = 64

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

print(f"Batches per epoch (train): {len(train_loader)}")
print(f"Batches per epoch (val): {len(val_loader)}")


In [ ]:
def train_model(model, criterion, optimizer, epochs=15, scheduler=None, early_stop_patience=None):
    train_acc_history, val_acc_history = [], []
    train_loss_history, val_loss_history = [], []

    best_val_loss = float('inf')
    patience_counter = 0
    best_model_state = None

    for epoch in range(epochs):
        # Training Phase
        model.train()
        running_loss = 0.0
        correct = 0
        total = 0

        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            optimizer.zero_grad()
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

        epoch_train_loss = running_loss / total
        epoch_train_acc = correct / total

        # Validation Phase
        model.eval()
        val_loss = 0.0
        correct = 0
        total = 0
        with torch.no_grad():
            for inputs, labels in val_loader:
                inputs, labels = inputs.to(device), labels.to(device)
                outputs = model(inputs)
                loss = criterion(outputs, labels)

                val_loss += loss.item() * inputs.size(0)
                _, predicted = outputs.max(1)
                total += labels.size(0)
                correct += predicted.eq(labels).sum().item()

        epoch_val_loss = val_loss / total
        epoch_val_acc = correct / total

        train_loss_history.append(epoch_train_loss)
        val_loss_history.append(epoch_val_loss)
        train_acc_history.append(epoch_train_acc)
        val_acc_history.append(epoch_val_acc)

        print(f"Epoch {epoch+1}/{epochs} | Train Loss: {epoch_train_loss:.4f} | Train Acc: {epoch_train_acc:.4f} | Val Loss: {epoch_val_loss:.4f} | Val Acc: {epoch_val_acc:.4f}")

        if scheduler:
            scheduler.step(epoch_val_loss)

        # Early Stopping Logic
        if early_stop_patience:
            if epoch_val_loss < best_val_loss:
                best_val_loss = epoch_val_loss
                patience_counter = 0
                best_model_state = model.state_dict()
            else:
                patience_counter += 1
                if patience_counter >= early_stop_patience:
                    print(f"Early stopping triggered at epoch {epoch+1}")
                    model.load_state_dict(best_model_state)
                    break

    return {
        'train_acc': train_acc_history, 'val_acc': val_acc_history,
        'train_loss': train_loss_history, 'val_loss': val_loss_history
    }

def evaluate_model(model, loader):
    model.eval()
    test_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []

    criterion = nn.CrossEntropyLoss()

    with torch.no_grad():
        for inputs, labels in loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)

            test_loss += loss.item() * inputs.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_targets.extend(labels.cpu().numpy())

    return test_loss/total, correct/total, np.array(all_preds), np.array(all_targets)


---
## 4. Part 1 - ANN (Fully Connected Network)

ANN treats each image as a **flat vector of 3072 values** (32x32x3). It cannot capture spatial relationships between neighbouring pixels, so we expect lower accuracy compared to CNN.

Architecture:
- Flatten -> Linear(3072, 512) -> ReLU -> Dropout(0.3) -> Linear(512, 256) -> ReLU -> Linear(256, 128) -> ReLU -> Dropout(0.2) -> Linear(128, 10)


In [ ]:
class ANNModel(nn.Module):
    def __init__(self):
        super(ANNModel, self).__init__()
        self.fc1 = nn.Linear(32 * 32 * 3, 512)
        self.drop1 = nn.Dropout(0.3)
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.drop2 = nn.Dropout(0.2)
        self.fc4 = nn.Linear(128, 10)

    def forward(self, x):
        x = x.view(-1, 32 * 32 * 3) # Flatten
        x = F.relu(self.fc1(x))
        x = self.drop1(x)
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = self.drop2(x)
        x = self.fc4(x) # No softmax needed as CrossEntropyLoss applies it internally
        return x

ann_model = ANNModel().to(device)
ann_criterion = nn.CrossEntropyLoss()
ann_optimizer = optim.Adam(ann_model.parameters(), lr=0.001)

# Count parameters
ann_params = sum(p.numel() for p in ann_model.parameters() if p.requires_grad)
print(f"ANN Model Total Trainable Params: {ann_params:,}")


### 4.1 Train the ANN
Training for 15 epochs.


In [ ]:
print("Training ANN...")
ann_history = train_model(ann_model, ann_criterion, ann_optimizer, epochs=15)


### 4.2 Evaluate ANN on Test Set


In [ ]:
ann_test_loss, ann_test_acc, ann_preds, ann_targets = evaluate_model(ann_model, test_loader)
print(f"ANN Test Loss    : {ann_test_loss:.4f}")
print(f"ANN Test Accuracy: {ann_test_acc:.4f}")


### 4.3 ANN - Learning Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(ann_history['train_acc'], label='Train Accuracy', marker='o', markersize=4)
ax1.plot(ann_history['val_acc'], label='Val Accuracy', marker='s', markersize=4)
ax1.set_title("ANN - Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(ann_history['train_loss'], label='Train Loss', marker='o', markersize=4)
ax2.plot(ann_history['val_loss'], label='Val Loss', marker='s', markersize=4)
ax2.set_title("ANN - Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


### 4.4 ANN - Confusion Matrix & Classification Report


In [ ]:
print("Classification Report (ANN):")
print(classification_report(ann_targets, ann_preds, target_names=class_names))


In [ ]:
cm_ann = confusion_matrix(ann_targets, ann_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_ann, annot=True, fmt='d', cmap='Blues',
            xticklabels=class_names, yticklabels=class_names)
plt.title("ANN - Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


---
## 5. Part 2 - CNN (Convolutional Neural Network)

Unlike ANN, CNN preserves the **spatial structure** of images using:
- **Convolutional layers** - learn local spatial patterns (edges, textures)
- **Pooling layers** - downsample feature maps, reduce computation
- **Batch Normalization** - stabilize training, allow higher learning rates

Architecture:  
Conv2D(32) -> BN -> Conv2D(32) -> MaxPool -> Drop(0.25) -> Conv2D(64) -> BN -> Conv2D(64) -> MaxPool -> Drop(0.25) -> Conv2D(128) -> BN -> MaxPool -> Flatten -> Linear(256) -> Drop(0.4) -> Linear(10)


In [ ]:
class CNNModel(nn.Module):
    def __init__(self):
        super(CNNModel, self).__init__()
        # Block 1
        self.conv1 = nn.Conv2d(3, 32, kernel_size=3, padding=1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 32, kernel_size=3, padding=1)
        self.pool1 = nn.MaxPool2d(2, 2)
        self.drop1 = nn.Dropout2d(0.25)

        # Block 2
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(64)
        self.conv4 = nn.Conv2d(64, 64, kernel_size=3, padding=1)
        self.pool2 = nn.MaxPool2d(2, 2)
        self.drop2 = nn.Dropout2d(0.25)

        # Block 3
        self.conv5 = nn.Conv2d(64, 128, kernel_size=3, padding=1)
        self.bn3 = nn.BatchNorm2d(128)
        self.pool3 = nn.MaxPool2d(2, 2)

        # Dense Layers
        # After 3 pools of 2x2, a 32x32 image becomes 4x4.
        self.fc1 = nn.Linear(128 * 4 * 4, 256)
        self.drop_fc = nn.Dropout(0.4)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        # Block 1
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.conv2(x))
        x = self.drop1(self.pool1(x))

        # Block 2
        x = F.relu(self.bn2(self.conv3(x)))
        x = F.relu(self.conv4(x))
        x = self.drop2(self.pool2(x))

        # Block 3
        x = F.relu(self.bn3(self.conv5(x)))
        x = self.pool3(x)

        # Flatten
        x = x.view(-1, 128 * 4 * 4)

        # Dense
        x = F.relu(self.fc1(x))
        x = self.drop_fc(x)
        x = self.fc2(x)
        return x

cnn_model = CNNModel().to(device)
cnn_criterion = nn.CrossEntropyLoss()
cnn_optimizer = optim.Adam(cnn_model.parameters(), lr=0.001)

# Count parameters
cnn_params = sum(p.numel() for p in cnn_model.parameters() if p.requires_grad)
print(f"CNN Model Total Trainable Params: {cnn_params:,}")


### 5.1 Train the CNN


In [ ]:
print("Training CNN...")
cnn_history = train_model(cnn_model, cnn_criterion, cnn_optimizer, epochs=15)


### 5.2 Evaluate CNN on Test Set


In [ ]:
cnn_test_loss, cnn_test_acc, cnn_preds, cnn_targets = evaluate_model(cnn_model, test_loader)
print(f"CNN Test Loss    : {cnn_test_loss:.4f}")
print(f"CNN Test Accuracy: {cnn_test_acc:.4f}")


### 5.3 CNN - Learning Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(cnn_history['train_acc'], label='Train Accuracy', marker='o', markersize=4)
ax1.plot(cnn_history['val_acc'], label='Val Accuracy', marker='s', markersize=4)
ax1.set_title("CNN - Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(cnn_history['train_loss'], label='Train Loss', marker='o', markersize=4)
ax2.plot(cnn_history['val_loss'], label='Val Loss', marker='s', markersize=4)
ax2.set_title("CNN - Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


### 5.4 CNN - Confusion Matrix & Classification Report


In [ ]:
print("Classification Report (CNN):")
print(classification_report(cnn_targets, cnn_preds, target_names=class_names))


In [ ]:
cm_cnn = confusion_matrix(cnn_targets, cnn_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_cnn, annot=True, fmt='d', cmap='Greens',
            xticklabels=class_names, yticklabels=class_names)
plt.title("CNN - Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


---
## 6. ANN vs CNN - Direct Comparison

### 6.1 Validation Accuracy & Loss Curves (Side by Side)


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# Accuracy comparison
ax1.plot(ann_history['val_acc'], label='ANN Val Acc', marker='o', markersize=4, linestyle='--')
ax1.plot(cnn_history['val_acc'], label='CNN Val Acc', marker='s', markersize=4)
ax1.set_title("Validation Accuracy - ANN vs CNN")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

# Loss comparison
ax2.plot(ann_history['val_loss'], label='ANN Val Loss', marker='o', markersize=4, linestyle='--')
ax2.plot(cnn_history['val_loss'], label='CNN Val Loss', marker='s', markersize=4)
ax2.set_title("Validation Loss - ANN vs CNN")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


### 6.2 Summary Table


In [ ]:
comparison = pd.DataFrame({
    "Model": ["ANN (Baseline)", "CNN (Baseline)"],
    "Test Accuracy": [f"{ann_test_acc:.4f}", f"{cnn_test_acc:.4f}"],
    "Test Loss": [f"{ann_test_loss:.4f}", f"{cnn_test_loss:.4f}"],
    "Total Params": [ann_params, cnn_params]
})
comparison


---
## 7. Training Strategy Upgrades

Now let's push the CNN further with:
1. **Data Augmentation** - random flips, rotations, shifts to reduce overfitting
2. **Learning Rate Scheduling** - reduce LR when validation loss plateaus (ReduceLROnPlateau)
3. **EarlyStopping** - stop training if no improvement for 5 epochs

These are standard tricks used in practice to get better performance from the same architecture.


### 7.1 Data Augmentation Setup


In [ ]:
# Define augmented transforms for training
train_transform_aug = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.RandomAffine(0, translate=(0.1, 0.1)),
    transforms.ToTensor(),
])

# Create new dataset and dataloader for augmented training
aug_train_dataset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=train_transform_aug)
# We use the same train/val split indices to be fair
aug_train_subset = torch.utils.data.Subset(aug_train_dataset, train_dataset.indices)
aug_train_loader = DataLoader(aug_train_subset, batch_size=batch_size, shuffle=True, num_workers=0)

# visualize some augmented versions of one image
fig, axes = plt.subplots(1, 5, figsize=(12, 3))
sample_idx = train_dataset.indices[0]
for i, ax in enumerate(axes):
    img, _ = aug_train_dataset[sample_idx]
    img = np.transpose(img.numpy(), (1, 2, 0))
    ax.imshow(img)
    ax.axis('off')
    ax.set_title(f"Aug {i+1}")
plt.suptitle("Augmented Versions of One Training Image", fontsize=13)
plt.tight_layout()
plt.show()


### 7.2 Build Enhanced CNN with Callbacks
We instantiate a new CNNModel but train it with augmented data, LR scheduling, and early stopping.


In [ ]:
enhanced_cnn = CNNModel().to(device)
enh_criterion = nn.CrossEntropyLoss()
enh_optimizer = optim.Adam(enhanced_cnn.parameters(), lr=0.001)

# Learning Rate Scheduler (reduces LR by factor of 0.5 if val_loss plateaus for 3 epochs)
lr_scheduler = optim.lr_scheduler.ReduceLROnPlateau(enh_optimizer, mode='min', factor=0.5, patience=3, verbose=True)

print("Enhanced CNN ready with EarlyStopping + ReduceLROnPlateau")


### 7.3 Train Enhanced CNN with Augmentation


In [ ]:
# To use aug_train_loader in our function, we temporarily mock train_loader
global train_loader
original_train_loader = train_loader
train_loader = aug_train_loader

print("Training Enhanced CNN...")
enh_history = train_model(
    enhanced_cnn, 
    enh_criterion, 
    enh_optimizer, 
    epochs=30, # Allow more epochs because we have early stopping
    scheduler=lr_scheduler, 
    early_stop_patience=5
)

# Restore original loader just in case
train_loader = original_train_loader


### 7.4 Evaluate Enhanced CNN


In [ ]:
enh_test_loss, enh_test_acc, enh_preds, enh_targets = evaluate_model(enhanced_cnn, test_loader)
print(f"Enhanced CNN Test Loss    : {enh_test_loss:.4f}")
print(f"Enhanced CNN Test Accuracy: {enh_test_acc:.4f}")


### 7.5 Enhanced CNN - Learning Curves


In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(enh_history['train_acc'], label='Train Accuracy', marker='o', markersize=3)
ax1.plot(enh_history['val_acc'], label='Val Accuracy', marker='s', markersize=3)
ax1.set_title("Enhanced CNN - Accuracy")
ax1.set_xlabel("Epoch")
ax1.set_ylabel("Accuracy")
ax1.legend()
ax1.grid(True)

ax2.plot(enh_history['train_loss'], label='Train Loss', marker='o', markersize=3)
ax2.plot(enh_history['val_loss'], label='Val Loss', marker='s', markersize=3)
ax2.set_title("Enhanced CNN - Loss")
ax2.set_xlabel("Epoch")
ax2.set_ylabel("Loss")
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()


### 7.6 Enhanced CNN - Classification Report & Confusion Matrix


In [ ]:
print("Classification Report (Enhanced CNN):")
print(classification_report(enh_targets, enh_preds, target_names=class_names))


In [ ]:
cm_enh = confusion_matrix(enh_targets, enh_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm_enh, annot=True, fmt='d', cmap='Oranges',
            xticklabels=class_names, yticklabels=class_names)
plt.title("Enhanced CNN - Confusion Matrix")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.tight_layout()
plt.show()


### 7.7 Sample Misclassified Images
Looking at what the enhanced CNN gets wrong helps us understand its weaknesses.


In [ ]:
wrong_idx = np.where(enh_preds != enh_targets)[0]
sample_wrong = np.random.choice(wrong_idx, min(10, len(wrong_idx)), replace=False)

fig, axes = plt.subplots(2, 5, figsize=(14, 6))
for i, ax in enumerate(axes.flat):
    idx = sample_wrong[i]
    img, _ = test_dataset[idx]
    img = np.transpose(img.numpy(), (1, 2, 0))
    ax.imshow(img)
    ax.set_title(f"True: {class_names[enh_targets[idx]]}\nPred: {class_names[enh_preds[idx]]}",
                 fontsize=9, color='red')
    ax.axis('off')
plt.suptitle("Misclassified Samples (Enhanced CNN)", fontsize=14)
plt.tight_layout()
plt.show()


---
## 8. Final Comparison - All Three Models


In [ ]:
final_comparison = pd.DataFrame({
    "Model": ["ANN (Baseline)", "CNN (Baseline)", "CNN + Augmentation + Callbacks"],
    "Test Accuracy": [f"{ann_test_acc:.4f}", f"{cnn_test_acc:.4f}", f"{enh_test_acc:.4f}"],
    "Test Loss": [f"{ann_test_loss:.4f}", f"{cnn_test_loss:.4f}", f"{enh_test_loss:.4f}"],
    "Key Features": [
        "Flatten -> Linear layers only",
        "Conv2d + BatchNorm + MaxPool + Dropout",
        "Augmentation + ReduceLROnPlateau + EarlyStopping"
    ]
})

print("=" * 90)
print("FINAL MODEL COMPARISON")
print("=" * 90)
print(final_comparison.to_string(index=False))
print("=" * 90)


In [ ]:
model_names = ["ANN", "CNN", "CNN+Aug"]
accs = [ann_test_acc, cnn_test_acc, enh_test_acc]
losses = [ann_test_loss, cnn_test_loss, enh_test_loss]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

colors = ['#e74c3c', '#3498db', '#2ecc71']

bars1 = ax1.bar(model_names, accs, color=colors, edgecolor='black', linewidth=0.8)
ax1.set_title("Test Accuracy Comparison", fontsize=13, fontweight='bold')
ax1.set_ylabel("Accuracy")
ax1.set_ylim(0, 1.0)
for bar, acc in zip(bars1, accs):
    ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{acc:.2%}", ha='center', fontsize=11, fontweight='bold')

bars2 = ax2.bar(model_names, losses, color=colors, edgecolor='black', linewidth=0.8)
ax2.set_title("Test Loss Comparison", fontsize=13, fontweight='bold')
ax2.set_ylabel("Loss")
for bar, loss in zip(bars2, losses):
    ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
             f"{loss:.4f}", ha='center', fontsize=11, fontweight='bold')

plt.tight_layout()
plt.show()


---
## 9. Key Observations & Analysis

### Why CNN >> ANN for Image Classification?

| Aspect | ANN | CNN |
|--------|-----|-----|
| **Input handling** | Flattens 32x32x3 into 3072 vector, loses spatial info | Preserves 2D spatial structure |
| **Feature extraction** | Learns global patterns only | Learns local to global features hierarchically |
| **Parameter efficiency** | More params for lower accuracy | Fewer params, much higher accuracy |
| **Translation invariance** | None - pixel position matters | Yes - detects features anywhere in the image |
| **Overfitting tendency** | High (too many params, no spatial priors) | Lower (weight sharing, pooling) |

### Effect of Training Strategies

1. **Data Augmentation** - reduced the train-val gap by exposing the model to shifted, flipped, rotated versions of training images. This is like giving the model more diverse training data without collecting new images
2. **ReduceLROnPlateau** - helped the optimizer converge more carefully when validation loss stopped improving fast
3. **EarlyStopping** - prevented unnecessary epochs and restored best model weights, saving time and avoiding overfitting

### Hardest Classes
Looking at the confusion matrices, **cat** and **dog** are typically the hardest classes. This makes sense because:
- They have similar body shapes at 32x32 resolution
- Both are four-legged animals with similar textures
- Fine-grained details (whiskers, snout shape) are hard to see at low resolution


---
## 10. Conclusion

- **ANN is not suitable for image classification** - it treats every pixel independently and cannot learn spatial patterns
- **CNN significantly outperforms ANN** because convolution layers preserve and exploit the 2D spatial structure of images
- **Training strategies push CNN further** - data augmentation, learning rate scheduling, and early stopping all help improve generalization
- The biggest accuracy jump was from ANN to CNN, confirming that **architecture choice matters more than training tricks** for image tasks
- For even higher accuracy, one could try:
  - Deeper architectures (ResNet, VGG, EfficientNet)
  - Transfer learning with pretrained weights
  - More aggressive augmentation (CutOut, MixUp)

**This assignment helped me understand the complete deep learning pipeline and why CNNs are the standard architecture for computer vision tasks.**
